# Nome do Aluno: Milton Salgado Leandro
# Instituição de Ensino: Universidade Federal do Rio de Janeiro


# Landing_to_Bronze — Ingestão da Camada Bronze

**Task no Workflow:** `to_Bronze`

**Objetivo:** ingerir os cinco CSVs do Volume e a cotação PTAX na camada Bronze, preservando os dados brutos e registrando `ingestion_datetime`.

**Estratégia para reexecução:** as tabelas continuam sendo gravadas em Delta com **Append**, porém uma tabela técnica de controle registra a versão de cada arquivo já processado. Assim, execuções recorrentes do Workflow não duplicam a mesma carga.

A limpeza e as regras de negócio permanecem restritas à camada Silver.


## 1.1 — Configuração do Ambiente

Define catálogo, Volume, Schemas e o mapeamento entre arquivos e tabelas. A tabela `bronze.tb_ingestion_control` é usada apenas para controle técnico da ingestão.


In [0]:
from datetime import date, timedelta
import requests

from pyspark.sql import functions as F

CATALOGO = "workspace"
BASE = f"/Volumes/{CATALOGO}/landing/inputs"

spark.sql(f"USE CATALOG {CATALOGO}")

spark.sql("CREATE SCHEMA IF NOT EXISTS landing")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql("CREATE VOLUME IF NOT EXISTS landing.inputs")

arquivos = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

spark.sql("""
    CREATE TABLE IF NOT EXISTS bronze.tb_ingestion_control (
        file_name STRING,
        file_path STRING,
        file_size BIGINT,
        modification_time BIGINT,
        target_table STRING,
        record_count BIGINT,
        ingestion_datetime TIMESTAMP
    )
    USING DELTA
""")


DataFrame[]

## 1.2 — Reset Opcional da Bronze

O parâmetro `reset_bronze` permite reconstruir a camada quando necessário.

- `false`: modo padrão do Workflow; preserva o histórico e processa apenas novas versões.
- `true`: limpa as tabelas deste projeto antes da carga.

Em execuções agendadas, mantenha `reset_bronze=false`.


In [0]:
try:
    dbutils.widgets.get("reset_bronze")
except:
    dbutils.widgets.dropdown(
        "reset_bronze",
        "false",
        ["false", "true"],
        "Reset Bronze"
    )

reset_bronze = (
    dbutils.widgets.get("reset_bronze")
    .strip()
    .lower()
    == "true"
)

if reset_bronze:
    print("RESET DA BRONZE SOLICITADO")

    tabelas_reset = list(arquivos.values()) + [
        "tb_cotacao_dolar",
        "tb_ingestion_control"
    ]

    for tabela in tabelas_reset:
        nome_completo = f"bronze.{tabela}"

        if spark.catalog.tableExists(nome_completo):
            spark.sql(f"TRUNCATE TABLE {nome_completo}")
            print(f"Limpa: {nome_completo}")

    print("Reset concluído.")
else:
    print("Reset desativado. Carga incremental habilitada.")


Reset desativado. Carga incremental habilitada.


## 1.3 — Validação dos Arquivos de Entrada

Confere se os cinco CSVs esperados estão disponíveis no Volume e valida a leitura linha a linha antes da ingestão. Uma falha interrompe `to_Bronze`, impedindo a execução das tarefas dependentes.


In [0]:
arquivos_volume = dbutils.fs.ls(BASE)
metadata_por_nome = {
    item.name: item
    for item in arquivos_volume
}

arquivos_esperados = set(arquivos.keys())
arquivos_encontrados = set(metadata_por_nome.keys())

faltantes = arquivos_esperados - arquivos_encontrados

if faltantes:
    raise RuntimeError(
        f"Arquivos ausentes no Volume: {sorted(faltantes)}"
    )

print("Todos os 5 CSVs foram encontrados.")

for arquivo, tabela in arquivos.items():
    caminho = f"{BASE}/{arquivo}"

    linhas_fisicas = (
        spark.read.text(caminho).count() - 1
    )

    df_teste = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", False)
        .option("mode", "PERMISSIVE")
        .csv(caminho)
    )

    total_lido = df_teste.count()

    ids_invalidos = (
        df_teste
        .filter(
            ~F.coalesce(
                F.col("id").rlike(r"^[0-9]+$"),
                F.lit(False)
            )
        )
        .count()
    )

    print(
        f"{tabela}: "
        f"{total_lido}/{linhas_fisicas} registros; "
        f"IDs inválidos={ids_invalidos}"
    )

    if total_lido != linhas_fisicas:
        raise RuntimeError(
            f"Divergência de leitura em {arquivo}: "
            f"{total_lido} != {linhas_fisicas}"
        )

    if ids_invalidos != 0:
        raise RuntimeError(
            f"IDs inválidos encontrados em {arquivo}"
        )


Todos os 5 CSVs foram encontrados.
tb_movies_info: 106930/106930 registros; IDs inválidos=0
tb_movies_financials: 106165/106165 registros; IDs inválidos=0
tb_movies_metrics: 107364/107364 registros; IDs inválidos=0
tb_credits_and_tags: 106320/106320 registros; IDs inválidos=0
tb_movies_reviews: 32412/32412 registros; IDs inválidos=0


## 1.4 — Ingestão Idempotente dos CSVs

Cada versão do arquivo é identificada por nome, tamanho e data de modificação. Se a mesma versão já constar em `tb_ingestion_control`, a carga é ignorada; caso contrário, os registros são adicionados à tabela Bronze em Delta/Append.


In [0]:
for arquivo, tabela in arquivos.items():
    caminho = f"{BASE}/{arquivo}"
    metadata = metadata_por_nome[arquivo]

    tamanho = int(metadata.size)
    modificacao = int(metadata.modificationTime)

    ja_processado = (
        spark.table("bronze.tb_ingestion_control")
        .filter(
            (F.col("file_name") == arquivo)
            & (F.col("file_size") == tamanho)
            & (F.col("modification_time") == modificacao)
        )
        .limit(1)
        .count()
        > 0
    )

    if ja_processado:
        print(f"SKIP: {arquivo} já foi processado.")
        continue

    print(f"PROCESSANDO: {arquivo}")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", False)
        .option("mode", "PERMISSIVE")
        .csv(caminho)
    )

    qtd_registros = df.count()

    df_bronze = df.withColumn(
        "ingestion_datetime",
        F.current_timestamp()
    )

    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"bronze.{tabela}")
    )

    controle = (
        spark.createDataFrame(
            [(
                arquivo,
                caminho,
                tamanho,
                modificacao,
                tabela,
                int(qtd_registros)
            )],
            [
                "file_name",
                "file_path",
                "file_size",
                "modification_time",
                "target_table",
                "record_count"
            ]
        )
        .withColumn(
            "ingestion_datetime",
            F.current_timestamp()
        )
    )

    (
        controle.write
        .format("delta")
        .mode("append")
        .saveAsTable("bronze.tb_ingestion_control")
    )

    print(
        f"OK: bronze.{tabela} "
        f"({qtd_registros} registros)"
    )


SKIP: movies_info_TMDB_IMDB.csv já foi processado.
SKIP: movies_financials_IMDB_TMDB.csv já foi processado.
SKIP: movies_metrics_IMDB_TMDB.csv já foi processado.
SKIP: credits_and_tags_IMDB_TMDB.csv já foi processado.
SKIP: movies_reviews.csv já foi processado.


## 1.5 — Cotação do Dólar via API PTAX

Os widgets `data_inicio` e `data_fim` podem ser enviados pelo Job. Quando vazios, o notebook consulta automaticamente os últimos sete dias corridos. Registros PTAX já existentes são descartados antes do Append.


In [0]:
try:
    dbutils.widgets.get("data_inicio")
except:
    dbutils.widgets.text(
        "data_inicio",
        "",
        "Data inicial (MM-DD-YYYY)"
    )

try:
    dbutils.widgets.get("data_fim")
except:
    dbutils.widgets.text(
        "data_fim",
        "",
        "Data final (MM-DD-YYYY)"
    )

inicio_param = dbutils.widgets.get("data_inicio").strip()
fim_param = dbutils.widgets.get("data_fim").strip()

inicio = (
    inicio_param
    if inicio_param
    else (
        date.today() - timedelta(days=7)
    ).strftime("%m-%d-%Y")
)

fim = (
    fim_param
    if fim_param
    else date.today().strftime("%m-%d-%Y")
)

print(f"Consulta PTAX: {inicio} até {fim}")

url = (
    "https://olinda.bcb.gov.br/olinda/"
    "servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo("
    "dataInicial=@dataInicial,"
    "dataFinalCotacao=@dataFinalCotacao)"
)

params = {
    "@dataInicial": f"'{inicio}'",
    "@dataFinalCotacao": f"'{fim}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json"
}

resposta = requests.get(
    url,
    params=params,
    timeout=30
)

resposta.raise_for_status()

dados = resposta.json().get("value", [])

if not dados:
    raise RuntimeError(
        "A API PTAX não retornou cotações para o período."
    )

df_cotacao = (
    spark.createDataFrame(dados)
    .select(
        "dataHoraCotacao",
        "cotacaoCompra"
    )
)


Consulta PTAX: 09-30-2026 até 10-07-2026


In [0]:
tabela_cotacao = "bronze.tb_cotacao_dolar"

# Verificar quais cotações já existem na Bronze.
if spark.catalog.tableExists(tabela_cotacao):

    cotacoes_existentes = (
        spark.table(tabela_cotacao)
        .select(
            "dataHoraCotacao",
            "cotacaoCompra"
        )
        .distinct()
    )

    df_novas_cotacoes = (
        df_cotacao
        .join(
            cotacoes_existentes,
            [
                "dataHoraCotacao",
                "cotacaoCompra"
            ],
            "left_anti"
        )
    )

else:
    df_novas_cotacoes = df_cotacao


# A API retorna poucos registros.
# Materializamos localmente para evitar que o
# DataFrame seja recalculado após o Append.
novas_cotacoes = (
    df_novas_cotacoes.collect()
)

qtd_novas = len(novas_cotacoes)


if qtd_novas > 0:

    # Recriar um DataFrame independente da
    # tabela Bronze usada no left_anti.
    df_para_gravar = spark.createDataFrame(
        novas_cotacoes,
        schema=df_cotacao.schema
    )

    df_para_gravar = (
        df_para_gravar
        .withColumn(
            "ingestion_datetime",
            F.current_timestamp()
        )
    )

    (
        df_para_gravar.write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela_cotacao)
    )

    print(
        f"{qtd_novas} novas "
        "cotações adicionadas."
    )

    display(
        df_para_gravar.limit(20)
    )

else:

    print(
        "Nenhuma nova cotação encontrada."
    )

Nenhuma nova cotação encontrada.


## 1.6 — Quality Gate da Bronze

Confirma a existência e o preenchimento das tabelas, a presença de `ingestion_datetime` e o registro da versão atual de cada CSV na tabela de controle. Qualquer falha interrompe a tarefa `to_Bronze`.


In [0]:
for arquivo, tabela in arquivos.items():
    nome = f"bronze.{tabela}"

    if not spark.catalog.tableExists(nome):
        raise RuntimeError(f"Tabela ausente: {nome}")

    df = spark.table(nome)

    if "ingestion_datetime" not in df.columns:
        raise RuntimeError(
            f"{nome} sem ingestion_datetime"
        )

    if df.count() == 0:
        raise RuntimeError(f"{nome} está vazia")

    metadata = metadata_por_nome[arquivo]

    controle_atual = (
        spark.table("bronze.tb_ingestion_control")
        .filter(
            (F.col("file_name") == arquivo)
            & (F.col("file_size") == int(metadata.size))
            & (
                F.col("modification_time")
                == int(metadata.modificationTime)
            )
        )
        .orderBy(
            F.col("ingestion_datetime").desc()
        )
        .limit(1)
        .collect()
    )

    if not controle_atual:
        raise RuntimeError(
            f"Versão atual não registrada: {arquivo}"
        )

    linhas_fisicas = (
        spark.read.text(
            f"{BASE}/{arquivo}"
        ).count() - 1
    )

    if int(controle_atual[0]["record_count"]) != linhas_fisicas:
        raise RuntimeError(
            f"Contagem registrada divergente em {arquivo}"
        )

    print(
        f"OK: {nome} | "
        f"versão atual={linhas_fisicas} registros"
    )

if not spark.catalog.tableExists(
    "bronze.tb_cotacao_dolar"
):
    raise RuntimeError(
        "Tabela bronze.tb_cotacao_dolar ausente"
    )

if (
    spark.table("bronze.tb_cotacao_dolar")
    .count()
    == 0
):
    raise RuntimeError(
        "Tabela bronze.tb_cotacao_dolar vazia"
    )

print("\nBRONZE VALIDADA COM SUCESSO!")


OK: bronze.tb_movies_info | versão atual=106930 registros
OK: bronze.tb_movies_financials | versão atual=106165 registros
OK: bronze.tb_movies_metrics | versão atual=107364 registros
OK: bronze.tb_credits_and_tags | versão atual=106320 registros
OK: bronze.tb_movies_reviews | versão atual=32412 registros

BRONZE VALIDADA COM SUCESSO!
